# ATLAS INSIGHT — Data Analyst AI
## Stage 1: Data Preparation & EDA | Stage 2: ML Models & UI

**Nhóm:** [Điền tên nhóm và MSSV]  
**Dataset:** Gapminder 1952–2007 (142 quốc gia, 1.704 country-year records).  
**BẮT BUỘC đọc:** `README.md`, `BAO_CAO_KIEM_TRA_YEU_CAU.md`.  

Các chỉ số `sim_*` là **giả lập**. Hai feed dữ liệu là hai phân nhóm từ **cùng một bộ dữ liệu gốc**, không được nói là 2 nhà xuất bản độc lập.

In [1]:
# ORIGINAL DATASET URL: https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv
from pathlib import Path
import sys,subprocess,json
import pandas as pd
from IPython.display import display,Image,Markdown
ROOT=Path.cwd() if (Path.cwd()/'src').is_dir() else Path.cwd().parent
print('Project root:',ROOT)
print('Public dataset URL: https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv')

Project root: f:\Final Project AI\AI_Data_Analyst_AI_Full_Project\AI_Data_Analyst_Project
Public dataset URL: https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv


## PHẦN I — GIAI ĐOẠN 1
### 1. Static Data Acquisition + Dynamic Synthetic Generation

Chọn dữ liệu nghiên cứu thật có truy xuất nguồn gốc. 2 feed theo châu lục (A=1.020, B=684), chuẩn hóa tên/kiểu cột và nối bằng `pd.concat`. Dữ liệu động tạo bằng `numpy.random` với seed cố định, tách biệt nguồn thật. `pd.merge` theo khóa quốc gia-năm, **left join** để không xóa các quan sát thật.

In [2]:
subprocess.run([sys.executable,str(ROOT/'src/01_prepare_data.py')],cwd=ROOT,check=True)
a=pd.read_csv(ROOT/'data/raw/Feed_A_Africa_Asia.csv')
b=pd.read_csv(ROOT/'data/raw/Feed_B_Other_Regions.csv')
sim=pd.read_csv(ROOT/'data/synthetic/simulated_country_year_indicators.csv')
master=pd.read_csv(ROOT/'data/processed/master_sparse.csv')
print('Feed sizes:',len(a),len(b),'| Synthetic:',len(sim),'| Left-merged:',len(master))
display(master.head(5))

CalledProcessError: Command '['c:\\Python311\\python.exe', 'f:\\Final Project AI\\AI_Data_Analyst_AI_Full_Project\\AI_Data_Analyst_Project\\src\\01_prepare_data.py']' returned non-zero exit status 1.

### 2. SQLite Storage → Data Retrieval → Cleaning & EDA

Thực hiện `master_sparse.to_sql`, đọc `SELECT *`, gán thiếu theo median theo châu lục, Unknown cho category, ffill/bfill trong từng chuỗi quốc gia. Xuất các bảng thống kê mô tả mean/median/std/skew/kurt và IQR.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/02_storage_clean_eda.py')],cwd=ROOT,check=True)
miss=pd.read_csv(ROOT/'reports/missingness_before.csv')
clean=pd.read_csv(ROOT/'data/processed/master_clean.csv')
print('Missing before:',miss.missing_count.sum(),'| after:',clean.isna().sum().sum())
display(pd.read_csv(ROOT/'reports/imputation_impact.csv'))
display(pd.read_csv(ROOT/'reports/univariate_statistics.csv').head())

### 3. EDA: tối thiểu 5 biểu đồ + giải thích “So What?”

#### 01 missingness

In [ ]:
display(Image(filename=str(ROOT/"charts/01_missingness.png"),width=760))

**SO WHAT?** Có 375 ô thiếu chỉ ở phần mô phỏng. Không bỏ hàng; bảo toàn dữ liệu gốc và gán giá trị theo loại biến.

#### 02 imputation distribution

In [ ]:
display(Image(filename=str(ROOT/"charts/02_imputation_distribution.png"),width=760))

**SO WHAT?** Mean của các chỉ số giả lập chỉ thay đổi khoảng 0,05–0,18%, nhưng phân phối có thể co hẹp khi gán median.

#### 03 life expectancy hist kde

In [ ]:
display(Image(filename=str(ROOT/"charts/03_life_expectancy_hist_kde.png"),width=760))

**SO WHAT?** Tuổi thọ trung bình khoảng 59,47 năm trên toàn bộ country-year, nhưng khác nhau theo châu lục và thời gian. Cần mô hình đa biến.

#### 04 life expectancy box iqr

In [ ]:
display(Image(filename=str(ROOT/"charts/04_life_expectancy_box_iqr.png"),width=760))

**SO WHAT?** IQR cho tuổi thọ không phát hiện ngoại lai, còn GDP/dân số có ngoại lai; chọn log1p cho GDP/dân số chứ không xóa hàng.

#### 05 continent bar

In [ ]:
display(Image(filename=str(ROOT/"charts/05_continent_bar.png"),width=760))

**SO WHAT?** Oceania chỉ 24 quan sát so với Africa 624. Diễn giải phải ghi rõ kích thước mẫu.

#### 06 correlation heatmap

In [ ]:
display(Image(filename=str(ROOT/"charts/06_correlation_heatmap.png"),width=760))

**SO WHAT?** Tương quan Spearman GDP/người–tuổi thọ ≈0,827; không suy diễn thành quan hệ nhân quả.

#### 07 gdp lifeexp scatter

In [ ]:
display(Image(filename=str(ROOT/"charts/07_gdp_lifeexp_scatter.png"),width=760))

**SO WHAT?** GDP và tuổi thọ đồng biến theo xu hướng phi tuyến; thử hồi quy tuyến tính và cây để so sánh.

#### 08 continent lifeexp grouped box

In [ ]:
display(Image(filename=str(ROOT/"charts/08_continent_lifeexp_grouped_box.png"),width=760))

**SO WHAT?** Tuổi thọ có khác biệt nhóm theo châu lục, do đó OneHotEncoder(continent) cho mô hình.

#### 09 continent lifeexp stacked bar

In [ ]:
display(Image(filename=str(ROOT/"charts/09_continent_lifeexp_stacked_bar.png"),width=760))

**SO WHAT?** Lớp ≥70 năm không cân đối theo vùng; kiểm tra ratio lớp và đánh giá F1 macro.

#### 10 historical trend

In [ ]:
display(Image(filename=str(ROOT/"charts/10_historical_trend.png"),width=760))

**SO WHAT?** Dataset kết thúc năm 2007; không diễn giải đồ thị hay mô hình như dự báo năm 2026.

## PHẦN II — GIAI ĐOẠN 2
### 4. PCA, K-Means, Regression, Classification

Chuẩn hóa trước PCA; chọn số PCs ít nhất đạt 90% phương sai trong khoảng 85–95%. K-Means K=2..8 có Elbow WCSS và Silhouette trên hai biểu diễn. Học có giám sát chia **theo quốc gia** để country không xuất hiện cả train và test.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/03_model_stage2.py')],cwd=ROOT,check=True)
metrics=json.loads((ROOT/'reports/model_results.json').read_text())
print(json.dumps(metrics,indent=2,ensure_ascii=False))
display(pd.read_csv(ROOT/'reports/kmeans_k_benchmark.csv').head(10))
display(pd.read_csv(ROOT/'reports/regression_benchmark.csv'))
display(pd.read_csv(ROOT/'reports/classification_benchmark.csv'))

#### 11 pca cumulative variance

In [ ]:
display(Image(filename=str(ROOT/"charts/11_pca_cumulative_variance.png"),width=760))

**SO WHAT?** 7 PCs giữ khoảng 91,26% phương sai; đủ ngưỡng 85–95%, giảm chiều mô tả.

#### 12 kmeans elbow silhouette

In [ ]:
display(Image(filename=str(ROOT/"charts/12_kmeans_elbow_silhouette.png"),width=760))

**SO WHAT?** K=2 PCA silhouette 0,288 > original 0,264 cùng K. Mức phân cụm vừa, không khẳng định nhóm tự nhiên chắc chắn.

#### 13 kmeans 2d clusters

In [ ]:
display(Image(filename=str(ROOT/"charts/13_kmeans_2d_clusters.png"),width=760))

**SO WHAT?** Nhãn cụm đã được map vào bản master; PC1/PC2 ở đây chỉ là phép chiếu để nhìn trực quan.

#### 14 regression actual vs predicted

In [ ]:
display(Image(filename=str(ROOT/"charts/14_regression_actual_vs_predicted.png"),width=760))

**SO WHAT?** LinearRegression thắng RMSE 6,169 năm và R² ≈0,778. Sai số vẫn còn, không dùng cho quyết định thật.

#### 15 regression residual plot

In [ ]:
display(Image(filename=str(ROOT/"charts/15_regression_residual_plot.png"),width=760))

**SO WHAT?** Kiểm tra phần dư để biết mô hình có thiên lệch theo mức tuổi thọ dự đoán hay không.

#### 16 confusion matrix

In [ ]:
display(Image(filename=str(ROOT/"charts/16_confusion_matrix.png"),width=760))

**SO WHAT?** LogisticRegression F1 macro ≈0,845; confusion matrix 222 TN, 30 FP, 15 FN, 81 TP. F1 thích hợp hơn accuracy đơn thuần với lớp mất cân bằng.

### 5. Lưu 2 database, chạy UI, đề xuất khoa học

SQLite đã nạp và đọc lại. ChromaDB là mã triển khai chờ cài/chạy trong máy nhóm. Gradio UI có thể chạy từ terminal, không nhúng server dài hạn trực tiếp trong Notebook.

In [ ]:
import sqlite3
with sqlite3.connect(ROOT/'databases/gapminder_project.sqlite') as c:
    print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",c))
    print(pd.read_sql('SELECT COUNT(*) as n FROM master_with_clusters',c))
print('For vector DB run: python src/04_chromadb_vector.py')
print('For web UI run: python ui/app.py')

**3 khuyến nghị dựa trên quan sát, không phải kết luận nhân quả:**

1. **Bổ sung dữ liệu độc lập cho nhóm lịch sử GDP thấp:** Cụm 0 có GDP/người trung vị ~1.368 USD và tuổi thọ TB ~50,86 năm, thấp hơn cụm 1 (~9.867 USD và ~69,85 năm).
2. **Dùng hồi quy tuyến tính làm baseline:** RMSE 6,17 thấp hơn cây 6,73; nên kiểm tra MAE theo vùng và các biến quan trắc thật trước sử dụng.
3. **Dùng xác suất, F1 macro khi truyền thông nhãn:** LogisticRegression F1 macro ~0,845, tốt hơn tree ~0,820; tránh khẳng định chắc chắn dự báo cho nước cụ thể.

**Công việc còn lại:** chạy ChromaDB, chụp screenshot browser Gradio thực, xác nhận hai feed có đáp ứng yêu cầu hai nguồn độc lập không, điền thông tin nhóm. Bonus web scraping/LLM có script nhưng chưa thực thi.